# Forecasting Ordini - Pipeline Completa

**Obiettivo**: Prevedere il volume giornaliero di ordini di un e-commerce utilizzando:
1. **XGBoost** e **LightGBM** (tree-based ML)
2. **Prophet** (time series decomposition)

**Dataset**: Online Retail (UCI ML Repository) - ~541K transazioni, periodo Dic 2010 - Dic 2011

---
## 0. Setup e Importazioni

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
import warnings
import os
import json

warnings.filterwarnings('ignore')

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import lightgbm as lgb
from prophet import Prophet
import joblib

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print('Tutte le librerie caricate correttamente.')

---
## 1. Configurazione

In [ ]:
FILE_PATH = os.path.join('online_retail', 'Online Retail.xlsx')

TRAIN_END_DATE = '2011-10-31'
VAL_END_DATE = '2011-11-15'
FORECAST_DAYS = 30

print(f'Dataset: {FILE_PATH}')
print(f'Esiste: {os.path.exists(FILE_PATH)}')

---
## 2. Caricamento Dati

In [ ]:
df = pd.read_excel(FILE_PATH, engine='openpyxl')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

print(f'Dati caricati: {df.shape[0]:,} righe x {df.shape[1]} colonne')
print(f'Periodo: {df["InvoiceDate"].min()} - {df["InvoiceDate"].max()}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

---
## 3. Pulizia Dati

In [ ]:
print(f'Righe iniziali: {len(df):,}')

df = df.dropna(subset=['InvoiceDate'])
print(f'Dopo rimozione date NaN: {len(df):,}')

df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
print(f'Dopo rimozione resi: {len(df):,}')

df = df[df['Quantity'] > 0]
print(f'Dopo filtro Quantity > 0: {len(df):,}')

df = df[df['UnitPrice'] > 0]
print(f'Dopo filtro UnitPrice > 0: {len(df):,}')

df['Revenue'] = df['Quantity'] * df['UnitPrice']
df['Date'] = df['InvoiceDate'].dt.date

---
## 4. Analisi Esplorativa (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

df.groupby('Date')['InvoiceNo'].nunique().plot(ax=axes[0, 0], title='Ordini Giornalieri', linewidth=0.8)
axes[0, 0].set_ylabel('N. Ordini')

df.groupby('Date')['Revenue'].sum().plot(ax=axes[0, 1], title='Revenue Giornaliero', linewidth=0.8, color='green')
axes[0, 1].set_ylabel('Revenue (GBP)')

df.groupby('Date')['Quantity'].sum().plot(ax=axes[1, 0], title='Quantita Giornaliera', linewidth=0.8, color='orange')
axes[1, 0].set_ylabel('Quantita')

df.groupby('Date')['CustomerID'].nunique().plot(ax=axes[1, 1], title='Clienti Unici Giornalieri', linewidth=0.8, color='red')
axes[1, 1].set_ylabel('N. Clienti')

plt.suptitle('Panoramica Serie Temporali', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
daily_orders_eda = df.groupby(pd.to_datetime(df['Date']))['InvoiceNo'].nunique()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

daily_orders_eda.hist(bins=30, ax=axes[0], edgecolor='black', alpha=0.7)
axes[0].set_title('Distribuzione Ordini Giornalieri')
axes[0].set_xlabel('N. Ordini')
axes[0].axvline(daily_orders_eda.mean(), color='red', linestyle='--', label=f'Media: {daily_orders_eda.mean():.1f}')
axes[0].legend()

day_labels = {0: 'Lun', 1: 'Mar', 2: 'Mer', 3: 'Gio', 4: 'Ven', 5: 'Sab', 6: 'Dom'}
avg_by_day = daily_orders_eda.groupby(daily_orders_eda.index.dayofweek).mean()
avg_by_day.index = [day_labels[d] for d in avg_by_day.index]
avg_by_day.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('Media Ordini per Giorno della Settimana')
axes[1].set_ylabel('Media Ordini')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

---
## 5. Aggregazione Giornaliera

In [ ]:
daily = df.groupby('Date').agg({
    'InvoiceNo': 'nunique',
    'Quantity': 'sum',
    'Revenue': 'sum',
    'CustomerID': 'nunique',
    'StockCode': 'nunique'
}).reset_index()

daily.columns = ['Date', 'Orders', 'TotalQuantity', 'Revenue', 'UniqueCustomers', 'UniqueProducts']
daily['Date'] = pd.to_datetime(daily['Date'])
daily = daily.sort_values('Date').reset_index(drop=True)

print(f'Giorni totali: {len(daily)}')
print(f'Media ordini/giorno: {daily["Orders"].mean():.1f}')
print(f'Min ordini: {daily["Orders"].min()}, Max ordini: {daily["Orders"].max()}')
daily.head(10)

---
## 6. Feature Engineering

In [ ]:
# Feature temporali
daily['Year'] = daily['Date'].dt.year
daily['Month'] = daily['Date'].dt.month
daily['Day'] = daily['Date'].dt.day
daily['DayOfWeek'] = daily['Date'].dt.dayofweek
daily['DayOfYear'] = daily['Date'].dt.dayofyear
daily['WeekOfYear'] = daily['Date'].dt.isocalendar().week
daily['Quarter'] = daily['Date'].dt.quarter

# Encoding ciclico
daily['DayOfWeek_sin'] = np.sin(2 * np.pi * daily['DayOfWeek'] / 7)
daily['DayOfWeek_cos'] = np.cos(2 * np.pi * daily['DayOfWeek'] / 7)
daily['Month_sin'] = np.sin(2 * np.pi * daily['Month'] / 12)
daily['Month_cos'] = np.cos(2 * np.pi * daily['Month'] / 12)

# Flag
daily['IsWeekend'] = (daily['DayOfWeek'] >= 5).astype(int)
daily['IsMonthStart'] = (daily['Day'] <= 5).astype(int)
daily['IsMonthEnd'] = (daily['Day'] >= 25).astype(int)

# Lag features
lags = [1, 2, 3, 7, 14, 28]
for lag in lags:
    daily[f'Orders_lag_{lag}'] = daily['Orders'].shift(lag)

# Rolling features
windows = [7, 14, 28]
for window in windows:
    daily[f'Orders_rolling_mean_{window}'] = daily['Orders'].shift(1).rolling(window=window).mean()
    daily[f'Orders_rolling_std_{window}'] = daily['Orders'].shift(1).rolling(window=window).std()

daily_clean = daily.dropna().reset_index(drop=True)

print(f'Feature temporali: 15')
print(f'Lag features: {len(lags)}')
print(f'Rolling features: {len(windows) * 2}')
print(f'Righe finali: {len(daily_clean)}')
print(f'Feature totali: {len(daily_clean.columns) - 2}')

---
## 7. Split Train / Validation / Test

In [ ]:
train_end = pd.to_datetime(TRAIN_END_DATE)
val_end = pd.to_datetime(VAL_END_DATE)

train = daily_clean[daily_clean['Date'] <= train_end].copy()
val = daily_clean[(daily_clean['Date'] > train_end) & (daily_clean['Date'] <= val_end)].copy()
test = daily_clean[daily_clean['Date'] > val_end].copy()

exclude_cols = ['Date', 'Orders']
feature_cols = [col for col in daily_clean.columns if col not in exclude_cols]

X_train, y_train = train[feature_cols], train['Orders']
X_val, y_val = val[feature_cols], val['Orders']
X_test, y_test = test[feature_cols], test['Orders']

print(f'Train: {len(train)} giorni ({train["Date"].min().date()} - {train["Date"].max().date()})')
print(f'Val:   {len(val)} giorni ({val["Date"].min().date()} - {val["Date"].max().date()})')
print(f'Test:  {len(test)} giorni ({test["Date"].min().date()} - {test["Date"].max().date()})')
print(f'Features: {len(feature_cols)}')

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=train['Date'], y=y_train, name='Train', mode='lines'))
fig.add_trace(go.Scatter(x=val['Date'], y=y_val, name='Validation', mode='lines'))
fig.add_trace(go.Scatter(x=test['Date'], y=y_test, name='Test', mode='lines'))
fig.update_layout(title='Split Temporale dei Dati', xaxis_title='Data', yaxis_title='Ordini', height=400)
fig.show()

---
## 8. Baseline Models

In [ ]:
def evaluate(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': f'{mape:.2f}%', 'R2': f'{r2:.4f}'}

baselines = []

# Baseline Mean
preds_mean = np.full(len(val), train['Orders'].mean())
baselines.append(evaluate(y_val, preds_mean, 'Baseline Mean'))

# Baseline Last Value
preds_last = np.full(len(val), train['Orders'].iloc[-1])
baselines.append(evaluate(y_val, preds_last, 'Baseline Last'))

# Baseline Moving Average 7d
preds_ma7 = np.full(len(val), train['Orders'].tail(7).mean())
baselines.append(evaluate(y_val, preds_ma7, 'Baseline MA(7)'))

pd.DataFrame(baselines)

---
## 9. XGBoost

In [ ]:
xgb_params = {
    'objective': 'reg:squarederror',
    'max_depth': 6,
    'learning_rate': 0.1,
    'n_estimators': 100,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42
}

model_xgb = xgb.XGBRegressor(**xgb_params)
model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

preds_xgb = model_xgb.predict(X_val)
res_xgb = evaluate(y_val, preds_xgb, 'XGBoost')

print('XGBoost - Validation Results:')
for k, v in res_xgb.items():
    if k != 'Model':
        print(f'  {k}: {v}')

---
## 10. LightGBM

In [ ]:
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'max_depth': 6,
    'learning_rate': 0.1,
    'n_estimators': 100,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'verbose': -1
}

model_lgb = lgb.LGBMRegressor(**lgb_params)
model_lgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=10, verbose=False)]
)

preds_lgb = model_lgb.predict(X_val)
res_lgb = evaluate(y_val, preds_lgb, 'LightGBM')

print('LightGBM - Validation Results:')
for k, v in res_lgb.items():
    if k != 'Model':
        print(f'  {k}: {v}')

---
## 11. Confronto Modelli

In [ ]:
all_results = baselines + [res_xgb, res_lgb]
results_df = pd.DataFrame(all_results)

rmse_xgb = float(res_xgb['RMSE'])
rmse_lgb = float(res_lgb['RMSE'])
best_model_name = 'XGBoost' if rmse_xgb < rmse_lgb else 'LightGBM'
best_model = model_xgb if rmse_xgb < rmse_lgb else model_lgb
best_preds = preds_xgb if rmse_xgb < rmse_lgb else preds_lgb

print(f'BEST MODEL: {best_model_name} (RMSE: {min(rmse_xgb, rmse_lgb):.2f})')
results_df

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(
    x=results_df['Model'],
    y=results_df['RMSE'],
    marker_color=['#ef553b' if r > rmse_lgb else '#00cc96' for r in results_df['RMSE']],
    text=[f'{r:.1f}' for r in results_df['RMSE']],
    textposition='outside'
))
fig.update_layout(title='Confronto RMSE tra Modelli', yaxis_title='RMSE', height=400)
fig.show()

---
## 12. Visualizzazioni Dettagliate

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=val['Date'], y=y_val.values, name='Valori Reali', mode='lines+markers'))
fig.add_trace(go.Scatter(x=val['Date'], y=best_preds, name=f'Predizioni {best_model_name}', mode='lines+markers', line=dict(dash='dash')))
fig.update_layout(title=f'Validation Set - {best_model_name}: Predizioni vs Reali', xaxis_title='Data', yaxis_title='Ordini', height=450)
fig.show()

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1][:15]
    top_features = [feature_cols[i] for i in indices]
    top_importances = importances[indices]

    fig = go.Figure(go.Bar(
        x=top_importances[::-1],
        y=top_features[::-1],
        orientation='h',
        marker_color='steelblue'
    ))
    fig.update_layout(title=f'Top 15 Feature Importance ({best_model_name})', xaxis_title='Importanza', height=500)
    fig.show()

---
## 13. Valutazione su Test Set

In [ ]:
preds_test = best_model.predict(X_test)
res_test = evaluate(y_test, preds_test, f'{best_model_name} (Test)')

print(f'{best_model_name} - Test Set Results:')
for k, v in res_test.items():
    if k != 'Model':
        print(f'  {k}: {v}')

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=train['Date'], y=y_train.values, name='Train', mode='lines', opacity=0.5))
fig.add_trace(go.Scatter(x=val['Date'], y=y_val.values, name='Validation', mode='lines'))
fig.add_trace(go.Scatter(x=val['Date'], y=best_preds, name=f'Pred Val ({best_model_name})', mode='lines', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=test['Date'], y=y_test.values, name='Test', mode='lines'))
fig.add_trace(go.Scatter(x=test['Date'], y=preds_test, name=f'Pred Test ({best_model_name})', mode='lines', line=dict(dash='dash')))
fig.update_layout(title='Vista Completa: Train + Validation + Test con Predizioni', xaxis_title='Data', yaxis_title='Ordini', height=500)
fig.show()

---
## 14. Analisi Residui

In [ ]:
residuals_val = y_val.values - best_preds
residuals_test = y_test.values - preds_test

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(best_preds, residuals_val, alpha=0.7, label='Validation')
axes[0].scatter(preds_test, residuals_test, alpha=0.7, label='Test')
axes[0].axhline(y=0, color='red', linestyle='--')
axes[0].set_xlabel('Predizioni')
axes[0].set_ylabel('Residui')
axes[0].set_title('Residui vs Predizioni')
axes[0].legend()

axes[1].hist(residuals_test, bins=15, edgecolor='black', alpha=0.7)
axes[1].set_title('Distribuzione Residui (Test)')
axes[1].set_xlabel('Residuo')

axes[2].plot(test['Date'], residuals_test, marker='o')
axes[2].axhline(y=0, color='red', linestyle='--')
axes[2].set_title('Residui nel Tempo (Test)')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## 15. Prophet - Time Series Forecasting

In [ ]:
df_prophet = df.copy()
daily_sales = df_prophet.groupby(df_prophet['InvoiceDate'].dt.date)['Quantity'].sum().reset_index()
daily_sales.columns = ['ds', 'y']
daily_sales['ds'] = pd.to_datetime(daily_sales['ds'])

print(f'Giorni di dati per Prophet: {len(daily_sales)}')
print(f'Periodo: {daily_sales["ds"].min()} - {daily_sales["ds"].max()}')

In [ ]:
model_prophet = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05
)
model_prophet.fit(daily_sales)
print('Prophet addestrato.')

In [ ]:
future = model_prophet.make_future_dataframe(periods=FORECAST_DAYS)
forecast = model_prophet.predict(future)

fig1 = model_prophet.plot(forecast)
plt.title(f'Prophet Forecast - Prossimi {FORECAST_DAYS} Giorni', fontsize=14, pad=20)
plt.ylabel('Quantita Ordinata')
plt.xlabel('Data')
plt.tight_layout()
plt.show()

In [ ]:
fig2 = model_prophet.plot_components(forecast)
plt.tight_layout()
plt.show()

### Pattern Settimanale e Annuale

In [ ]:
giorni = ['Lunedi', 'Martedi', 'Mercoledi', 'Giovedi', 'Venerdi', 'Sabato', 'Domenica']
week_dates = pd.date_range('2023-01-02', periods=7)
week_df = pd.DataFrame({'ds': week_dates})
weekly_effect = model_prophet.predict_seasonal_components(model_prophet.setup_dataframe(week_df))['weekly'].values

fig = go.Figure(go.Bar(
    x=giorni,
    y=weekly_effect,
    marker_color=['#00cc96' if e > 0 else '#ef553b' for e in weekly_effect],
    text=[f'{e:+.0f}' for e in weekly_effect],
    textposition='outside'
))
fig.update_layout(title='Prophet - Pattern Settimanale', yaxis_title='Effetto (unita vs media)', height=400)
fig.show()

print(f'Giorno MIGLIORE: {giorni[np.argmax(weekly_effect)]} ({weekly_effect.max():+.0f} unita)')
print(f'Giorno PEGGIORE: {giorni[np.argmin(weekly_effect)]} ({weekly_effect.min():+.0f} unita)')

In [ ]:
mesi = ['Gen', 'Feb', 'Mar', 'Apr', 'Mag', 'Giu', 'Lug', 'Ago', 'Set', 'Ott', 'Nov', 'Dic']
year_dates = pd.date_range('2023-01-01', periods=365)
year_df = pd.DataFrame({'ds': year_dates})
yearly_effect = model_prophet.predict_seasonal_components(model_prophet.setup_dataframe(year_df))['yearly'].values

monthly_effect = []
for i in range(12):
    month_start = i * 30
    month_end = min((i + 1) * 30, len(yearly_effect))
    monthly_effect.append(yearly_effect[month_start:month_end].mean())

fig = go.Figure(go.Bar(
    x=mesi,
    y=monthly_effect,
    marker_color=['#00cc96' if e > 0 else '#ef553b' for e in monthly_effect],
    text=[f'{e:+.0f}' for e in monthly_effect],
    textposition='outside'
))
fig.update_layout(title='Prophet - Pattern Annuale', yaxis_title='Effetto (unita vs media)', height=400)
fig.show()

print(f'Mese MIGLIORE: {mesi[np.argmax(monthly_effect)]} ({max(monthly_effect):+.0f} unita)')
print(f'Mese PEGGIORE: {mesi[np.argmin(monthly_effect)]} ({min(monthly_effect):+.0f} unita)')

### Previsioni Prossimi 7 Giorni

In [ ]:
next_week = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(7).copy()
next_week['ds'] = next_week['ds'].dt.strftime('%Y-%m-%d')
next_week.columns = ['Data', 'Previsione', 'Min (CI 80%)', 'Max (CI 80%)']

print('Previsioni Prophet - Prossimi 7 Giorni:')
print(next_week.to_string(index=False))

avg_fc = next_week['Previsione'].mean()
print(f'\nMedia prevista: {avg_fc:.0f} unita/giorno')
print(f'Safety stock suggerito: {next_week["Max (CI 80%)"].mean() * 1.1:.0f} unita/giorno (+10%)')

---
## 16. Salvataggio Modello e Artefatti

In [ ]:
model_filename = f'model_{best_model_name.lower()}.pkl'
joblib.dump(best_model, model_filename)
print(f'Modello salvato: {model_filename}')

with open('feature_columns.json', 'w') as f:
    json.dump(feature_cols, f)
print('Feature columns salvate: feature_columns.json')

---
## 17. Riepilogo Finale

In [ ]:
print('=' * 60)
print('RIEPILOGO FINALE')
print('=' * 60)

print(f'\nMiglior Modello ML: {best_model_name}')
print(f'  Validation RMSE: {min(rmse_xgb, rmse_lgb):.2f}')
print(f'  Test RMSE: {float(res_test["RMSE"]):.2f}')
print(f'  Test R2: {res_test["R2"]}')
print(f'  Test MAPE: {res_test["MAPE"]}')

trend_change = forecast['trend'].iloc[-1] - forecast['trend'].iloc[0]
trend_pct = (trend_change / forecast['trend'].iloc[0]) * 100
print(f'\nProphet - Trend: {"Crescita" if trend_change > 0 else "Decrescita"} ({trend_pct:+.1f}%)')
print(f'Prophet - Forecast medio 7gg: {avg_fc:.0f} unita/giorno')

print(f'\nFile generati:')
print(f'  - {model_filename}')
print(f'  - feature_columns.json')
print('\nProcesso completato.')